# SceneCritic on a Colab GPU
Select **Runtime → Change runtime type → GPU** first. This runs a text-only Qwen model directly on the GPU, without paid inference API calls. The default 4B model differs from the original hosted 27B model; their scene outputs will differ. A 16 GB GPU may run out of memory on long prompts with eager attention; use a higher-memory GPU or reduce `SCENECRITIC_MAX_TOKENS`.

Upload a ZIP of the prepared repository and extract it into `/content/SceneCritic`, or clone your own GitHub repository there. Never upload `.env` to GitHub. Public model weights do not need an HF token. This notebook is unexecuted; GPU inference must be verified in your runtime.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/content/SceneCritic')
assert (ROOT / 'requirements-colab.txt').exists(), 'Clone or extract the prepared repository to /content/SceneCritic first'
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.6'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '.venv'], check=True)
PYTHON = str(ROOT / '.venv/bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-colab.txt'], check=True)
# run_scene records package versions using pip.
subprocess.run(['uv', 'pip', 'install', '--python', PYTHON, 'pip==25.3'], check=True)


Resolve the model revision once, then copy the printed SHA into `REVISION` for every later local and Colab run. Use the same model, revision, dependencies, seed, inputs and GPU type for comparisons.

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
REVISION = ''  # Paste an immutable model commit SHA here for repeat runs.
if not REVISION:
    REVISION = subprocess.check_output([PYTHON, '-c',
        'from huggingface_hub import model_info; import sys; print(model_info(sys.argv[1]).sha)', MODEL], text=True).strip()
print('Model:', MODEL, 'Revision:', REVISION)
os.environ.update(SCENECRITIC_BACKEND='local', SCENECRITIC_MODEL=MODEL,
                  SCENECRITIC_API_BASE='http://127.0.0.1:7501/v1',
                  SCENECRITIC_SEED='42', SCENECRITIC_MAX_TOKENS='4096')
log = open('/content/scenecritic-server.log', 'w')
server = subprocess.Popen([PYTHON, 'scripts/local_model_server.py', '--model', MODEL,
                           '--revision', REVISION, '--max-new-tokens', '4096'],
                          stdout=log, stderr=subprocess.STDOUT)
print('Server PID:', server.pid, 'Logs: /content/scenecritic-server.log')


In [ ]:
# Run this cell until model loading has finished, then continue.
import urllib.request, json
print(json.load(urllib.request.urlopen('http://127.0.0.1:7501/v1/models', timeout=5)))


In [ ]:
# Each run must use a fresh output directory.
for name in ('run1', 'run2'):
    subprocess.run([PYTHON, 'scripts/run_scene.py', '--output', f'outputs/{name}', '--seed', '42'], check=True)
a = json.loads(Path('outputs/run1/scene_pos_0_0.json').read_text())
b = json.loads(Path('outputs/run2/scene_pos_0_0.json').read_text())
assert a == b, 'Scene JSON differs: inspect logs, manifests and server metadata'
print('Both generated scene JSONs match.')


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/scenecritic-results', 'zip', 'outputs')
files.download('/content/scenecritic-results.zip')
